###CUAD Contract Analyzer Project

This cell installs the necessary libraries for the CUAD data pipeline and restarts the notebook so the updated packages are available. 

In [0]:
%pip install -U "datasets==2.18.0" "huggingface_hub==0.23.5" pandas numpy sentence-transformers faiss-cpu langchain langchain-text-splitters
dbutils.library.restartPython()

This cell imports the libraries used for data loading, preprocessing, chunking, embeddings, vector indexing and artifact saving. 

In [0]:
import pandas as pd 
import numpy as np 
import faiss
import pickle
import re
import random

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

Below, the CUAD dataset is loaded from Hugging Face. CUAD is used as the legal contract corpus for the retrieval component of the project. 

In [0]:
dataset = load_dataset("theatticusproject/cuad")
dataset

The next few cells convert the dataset from Hugging Face into a pandas DataFrame and checks the dataset structure. This includes row count and available columns.

In [0]:
df = dataset["train"].to_pandas()

df.head()

In [0]:
df.shape

In [0]:
df.columns

The next list of cells calculate text length to understand the quality of the raw records. The results show that many records are empty or very short, so filtering is necessary before chunking and embeddings. 

In [0]:
df["text_length"] = df["text"].astype(str).apply(len)
df[["text", "text_length"]].head()

In [0]:
df["text_length"].describe()

Thess cells standardize the contract text by removing extra whitespace, tabs and line breaks. Cleaning the text improves chunk quality and retrieval consistency. 

In [0]:

def clean_text(text):
    text = str(text)
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")
    text = re.sub(r"\s+", " ", text)           
    return text.strip()

df["clean_text"] = df["text"].apply(clean_text)

df[["clean_text", "text_length"]].head()

In [0]:
df = df[df["clean_text"].str.len() > 20]
print("Rows remaining:", len(df))

The next few cells manually inspect sample records to confirm whether the remaining data contains real contract content rather than dataset information.

In [0]:
for i in range(10):
    print(f"\n----- ROW {i} -----")
    print(df.iloc[i]["clean_text"][:1000])

In [0]:
df["clean_text"].str.len().describe()

In [0]:
print(df.iloc[100]["clean_text"])

In [0]:
print(df.iloc[1000]["clean_text"])

In [0]:
print(df.iloc[5000]["clean_text"])

In [0]:
print(df.iloc[10000]["clean_text"])

In [0]:
df.sort_values("text_length", ascending=False)[["clean_text","text_length"]].head(10)

This cell removes records shorter than 50 characters because many short records are headers, blank rows or non substantive text that would not help retrieval. 

In [0]:
df = df[
    (df["clean_text"].str.len() > 50)
]

print("Rows Remaining:", len(df))

Here we remove the obvious dataset metadata and documentation rows, such as README text and answer format labels. This helps make sure the vector index focuses on actual contract language. 

In [0]:
metadata_patterns = [
    "CONTRACT UNDERSTANDING ATTICUS DATASET",
    "CUAD",
    "README",
    "Answer Format:",
]

for pattern in metadata_patterns:
    df = df[~df["clean_text"].str.contains(pattern, case=False, na=False)]

print("Rows remaining after metadata removal:", len(df))

In [0]:
df_clean = df.copy()
df_clean.shape

This cell removes records taht begin with "Source:" because these are document references rather than useful contract clauses. 

In [0]:
df_clean = df_clean[
    ~df_clean["clean_text"].str.startswith("Source:", na=False)
]

print("Rows after source removal:", len(df_clean))

In [0]:
df_clean.sample(10)[["clean_text","text_length"]]

This cell splits cleaned contract text into overlapping chunks. A chunk size of 750 with 150 overlap keeps
enough legal context while making the chunks small enough for retrieval.

In [0]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=750,
    chunk_overlap=150
)

chunks = []

for text in df_clean["clean_text"]:
    split_chunks = text_splitter.split_text(text)
    chunks.extend(split_chunks)

print("Total chunks:", len(chunks))

In [0]:
print(chunks[0])
print("\n\n")
print(chunks[1])
print("\n\n")
print(chunks[2])

This cell removes remaining CUAD documentation rows related to dataset files, categories and formats. These records were identified during chunk inspection and removed before final embedding.

In [0]:
df_clean = df_clean[
    ~df_clean["clean_text"].str.contains("master clauses CSV|SQuAD-style JSON|28 Excels|Category List", case=False, na=False)
]

print("Rows after documentation cleanup:", len(df_clean))

This cell splits cleaned contract text into overlapping chunks. A chunk size of 750 with 150 overlap keeps
enough legal context while making the chunks small enough for retrieval.

In [0]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=750,
    chunk_overlap=150
)

chunks = []

for text in df_clean["clean_text"]:
    split_chunks = text_splitter.split_text(text)
    chunks.extend(split_chunks)

print("Total chunks:", len(chunks))

This cell samples random chunks to verify that the final chunks contain useful legal language such as
confidentiality, termination, indemnification and other contract provisions.

In [0]:
for _ in range(5):
    idx = random.randint(0, len(chunks)-1)
    print(chunks[idx][:1000])
    print("\n")

This cell loads the Sentence Transformer embedding model used to convert contract chunks into numerical vectors for semantic search.

In [0]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

This cells generate embeddings for all cleaned contract chunks. Each chunk is converted into a 384 dimensional vector representation.

In [0]:
embeddings = model.encode(
    chunks,
    show_progress_bar=True
)

In [0]:
embeddings.shape

In [0]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(embeddings).astype("float32")
)

print("Vectors indexed:", index.ntotal)

This cell creates a FAISS vector index and stores all chunk embeddings inside it. This allows fast
semantic retrieval over the CUAD contract corpus.

In [0]:
faiss.write_index(
    index,
    "cuad_faiss_index.index"
)

print("FAISS index saved.")

This cell saves the FAISS index, chunks and embeddings so the agent notebook can reuse the retrieval pipeline without rebuilding it from scratch.

In [0]:
with open("cuad_chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

print("Chunks saved.")

In [0]:
np.save(
    "cuad_embeddings.npy",
    embeddings
)

print("Embeddings saved.")

These cells test semantic retrieval using contract related queries. The retrieved results are inspected to
confirm that the vector search returns relevant legal clauses.

In [0]:
query = "termination clause"

query_embedding = model.encode([query])

distances, indices = index.search(
    np.array(query_embedding).astype("float32"),
    k=5
)

for idx in indices[0]:
    print("\n")
    print(chunks[idx][:1000])

In [0]:
query = "confidentiality agreement"

query_embedding = model.encode([query])

distances, indices = index.search(
    np.array(query_embedding).astype("float32"),
    k=5
)

for idx in indices[0]:
    print("\n")
    print(chunks[idx][:1000])

In [0]:
query = "indemnification"

query_embedding = model.encode([query])

distances, indices = index.search(
    np.array(query_embedding).astype("float32"),
    k=5
)

for idx in indices[0]:
    print("\n")
    print(chunks[idx][:1000])

In [0]:
query = "termination clause"

query_embedding = model.encode([query])

distances, indices = index.search(
    np.array(query_embedding).astype("float32"),
    k=5
)

for rank, (idx, distance) in enumerate(zip(indices[0], distances[0]), start=1):
    print(f"\nResult {rank}")
    print("Distance: {distance}")
    print(chunks[idx][:500])

This function wraps the retrieval logic into a reusable tool. The agent notebook can call this function to
retrieve relevant CUAD clauses based on a user question.

In [0]:
def search_contracts(query, k=5):
    query_embedding = model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding).astype("float32"),
        k=k
    )

    results = []

    for idx, distance in zip(indices[0], distances[0]):
        results.append({
            "distance": float(distance),
            "text": chunks[idx]
        })

    return results

In [0]:
results = search_contracts("termination clause")

for r in results:
    print("\nDistance:", r["distance"])
    print(r["text"][:500])

This final summary confirms that the full data pipeline ran successfully. This includes cleaning, chunking,
embedding generation, FAISS indexing and retrieval validation.

In [0]:
print("CUAD Pipeline Summary\n")
print("Contracts after cleaning:", len(df_clean))
print("Total chunks:", len(chunks))
print("Embedding dimensions:", embeddings.shape[1])
print("Vectors indexed:", index.ntotal)